# 🔗 MCP with the OpenAI Responses API
### Hosted Tool Calling — Build Once, Call from Anywhere

> **What you'll learn:** How to expose a FastMCP server as a remotely-accessible tool server, then drive it from the OpenAI **Responses API** — OpenAI's modern API that has native, first-class MCP support.

---

## How This Differs from Function Calling

In Session 1 you learned **function calling** — you define JSON schemas in every API request, execute the functions yourself, and feed the results back in a loop.

The Responses API's **hosted MCP tool** works differently:

```
FUNCTION CALLING (Chat Completions):          HOSTED MCP (Responses API):

Your code defines schemas per-request         You point to a URL once
         ↓                                              ↓
Model asks to call a function                 OpenAI fetches tool list from MCP server
         ↓                                              ↓
Your code runs the function                   OpenAI calls the MCP server directly
         ↓                                              ↓
Your code feeds result back                   Result returned to model automatically
         ↓                                              ↓
Model answers                                 Model answers
```

With hosted MCP, **OpenAI's infrastructure talks to your MCP server directly** — no tool execution loop in your code.

## What You'll Build

```
┌────────────────────────┐        ┌──────────────────────┐        ┌──────────────────┐
│  Your Python Notebook  │        │   OpenAI Responses   │        │  Your FastMCP    │
│                        │        │        API           │        │    Server        │
│  client.responses      │──────► │                      │──────► │  (public URL)    │
│    .create(tools=[     │        │  1. Fetches tools    │        │                  │
│      {type: "mcp",     │        │  2. Calls tools      │        │  • roll_dice     │
│       server_url: ...  │        │  3. Returns answer   │        │  • get_weather   │
│      }])               │        │                      │        │  • analyze_text  │
│                        │◄────── │  output_text: "..."  │◄────── │  • convert_units │
└────────────────────────┘        └──────────────────────┘        └──────────────────┘
```

**Sections:**
1. 🔧 The FastMCP Server — building tools for remote hosting
2. 🌐 Deploying with ngrok — making your server reachable
3. 🚀 The Responses API — your first hosted MCP call
4. ⚙️ API Features — `allowed_tools`, `require_approval`, `previous_response_id`
5. 🔄 Multi-turn Conversations — state across requests
6. 🔐 Authentication — JWT-protected servers
7. 🏭 Production Patterns — multiple servers, mixing with function calling

---

### Prerequisites
- An OpenAI API key with Responses API access
- `fastmcp` and `openai` installed (cell below)
- `ngrok` for local development tunneling ([ngrok.com](https://ngrok.com) — free account)

In [1]:
# !pip install fastmcp openai --quiet

import os, json, time, subprocess, threading
from openai import OpenAI

openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
print("✅ Ready!")

✅ Ready!


---

## 🔧 Section 1: The FastMCP Server

The Responses API requires a **remotely accessible HTTP server** — it cannot connect to a local process or an in-memory server. The server must speak the **Streamable HTTP** or **SSE** transport.

### Key Constraint

> The Responses API only accesses **tools** from your MCP server. Resources and prompts are not surfaced. Design your server accordingly — put everything the model needs to act on into tools.

### What Makes a Good Responses API Tool
- **Clear docstring** — OpenAI passes this directly to the model as the tool description
- **Typed parameters** — FastMCP generates the JSON schema automatically
- **Self-contained** — each tool call should do one thing and return a complete result
- **No side-effects without intent** — mark destructive tools with `destructiveHint: True` in annotations

In [ ]:
%%writefile mcp_server.py
"""
mcp_server.py — A FastMCP server designed for the OpenAI Responses API.

Run this as a standalone HTTP server:
    python mcp_server.py

Then expose it publicly with ngrok:
    ngrok http 8000
"""

import random, math, re
from typing import Annotated, Literal
from pydantic import Field
from fastmcp import FastMCP

mcp = FastMCP(
    name="UtilityServer",
    instructions="A collection of utility tools: dice rolling, weather (simulated), "
                 "text analysis, and unit conversion.",
)


# ── Tool 1: Dice roller ────────────────────────────────────────────────────────
@mcp.tool
def roll_dice(
    n_dice: Annotated[int, Field(description="Number of dice to roll", ge=1, le=20)],
    sides: Annotated[int, Field(description="Number of sides per die", ge=2, le=100)] = 6,
) -> dict:
    """Roll n_dice dice each with the given number of sides. Returns individual rolls and total."""
    rolls = [random.randint(1, sides) for _ in range(n_dice)]
    return {"rolls": rolls, "total": sum(rolls), "n_dice": n_dice, "sides": sides}


# ── Tool 2: Weather (simulated — replace with a real API in production) ─────────
@mcp.tool
def get_weather(
    city: Annotated[str, "Name of the city"],
    units: Annotated[Literal["celsius", "fahrenheit"], "Temperature unit"] = "celsius",
) -> dict:
    """
    Get current weather conditions for a city.
    Returns temperature, condition, humidity, and wind speed.
    Note: This server returns simulated data for demonstration purposes.
    """
    # Simulated data — in production replace with httpx call to a weather API
    base_data = {
        "manila":    {"temp_c": 32, "condition": "Partly Cloudy",  "humidity": 78, "wind_kph": 15},
        "tokyo":     {"temp_c": 18, "condition": "Overcast",        "humidity": 62, "wind_kph": 20},
        "london":    {"temp_c": 12, "condition": "Rainy",           "humidity": 85, "wind_kph": 25},
        "new york":  {"temp_c": 22, "condition": "Sunny",           "humidity": 55, "wind_kph": 18},
        "singapore": {"temp_c": 30, "condition": "Thunderstorm",   "humidity": 90, "wind_kph": 10},
        "sydney":    {"temp_c": 24, "condition": "Clear",           "humidity": 60, "wind_kph": 22},
    }
    data = base_data.get(city.lower(), {"temp_c": 20, "condition": "Unknown", "humidity": 60, "wind_kph": 10})
    temp = data["temp_c"] if units == "celsius" else round(data["temp_c"] * 9/5 + 32, 1)
    return {
        "city": city.title(),
        "temperature": temp,
        "units": units,
        "condition": data["condition"],
        "humidity_pct": data["humidity"],
        "wind_kph": data["wind_kph"],
    }


# ── Tool 3: Text analysis ─────────────────────────────────────────────────────
@mcp.tool
def analyze_text(text: str) -> dict:
    """
    Analyze a piece of text and return detailed statistics.
    Returns character count, word count, sentence count, paragraph count,
    reading time estimate, and the top 5 most frequent words.
    """
    from collections import Counter
    words = re.findall(r"\b[a-zA-Z]+\b", text.lower())
    sentences = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    stopwords = {"the","a","an","is","are","was","were","and","or","but","in","on","at","to","of","for"}
    content_words = [w for w in words if w not in stopwords]
    reading_time_s = round(len(words) / (200 / 60))  # avg 200 wpm
    return {
        "character_count": len(text),
        "word_count": len(words),
        "sentence_count": len(sentences),
        "paragraph_count": len(paragraphs),
        "estimated_reading_seconds": reading_time_s,
        "top_5_words": Counter(content_words).most_common(5),
        "avg_words_per_sentence": round(len(words) / len(sentences), 1) if sentences else 0,
    }


# ── Tool 4: Unit conversion ────────────────────────────────────────────────────
@mcp.tool
def convert_units(
    value: Annotated[float, "The numeric value to convert"],
    from_unit: Annotated[str, "Source unit (e.g. km, kg, usd, celsius)"],
    to_unit: Annotated[str, "Target unit (e.g. miles, lbs, php, fahrenheit)"],
) -> dict:
    """
    Convert a value between units. Supports:
    - Length: km ↔ miles, m ↔ ft, cm ↔ inches
    - Weight: kg ↔ lbs, g ↔ oz
    - Temperature: celsius ↔ fahrenheit
    - Currency (approx.): usd ↔ php, usd ↔ eur, usd ↔ sgd
    """
    conversions = {
        ("km",       "miles"):      lambda v: v * 0.621371,
        ("miles",    "km"):         lambda v: v / 0.621371,
        ("m",        "ft"):         lambda v: v * 3.28084,
        ("ft",       "m"):          lambda v: v / 3.28084,
        ("cm",       "inches"):     lambda v: v / 2.54,
        ("inches",   "cm"):         lambda v: v * 2.54,
        ("kg",       "lbs"):        lambda v: v * 2.20462,
        ("lbs",      "kg"):         lambda v: v / 2.20462,
        ("g",        "oz"):         lambda v: v / 28.3495,
        ("oz",       "g"):          lambda v: v * 28.3495,
        ("celsius",  "fahrenheit"): lambda v: v * 9/5 + 32,
        ("fahrenheit","celsius"):   lambda v: (v - 32) * 5/9,
        ("usd",      "php"):        lambda v: v * 56.20,
        ("php",      "usd"):        lambda v: v / 56.20,
        ("usd",      "eur"):        lambda v: v * 0.92,
        ("eur",      "usd"):        lambda v: v / 0.92,
        ("usd",      "sgd"):        lambda v: v * 1.34,
        ("sgd",      "usd"):        lambda v: v / 1.34,
    }
    key = (from_unit.lower(), to_unit.lower())
    fn = conversions.get(key)
    if not fn:
        return {"error": f"Conversion from '{from_unit}' to '{to_unit}' not supported.",
                "supported_pairs": [f"{a} → {b}" for a, b in conversions]}
    result = fn(value)
    return {"original": value, "from": from_unit, "result": round(result, 4), "to": to_unit}


# ── Tool 5: Calculator ────────────────────────────────────────────────────────
@mcp.tool
def calculate(expression: str) -> dict:
    """
    Evaluate a mathematical expression. Supports arithmetic, sqrt(), abs(),
    round(), log(), sin(), cos(), tan(), pi, e.
    Examples: "2**10", "sqrt(144) + 50", "round(pi, 4)"
    """
    safe_globals = {
        "sqrt": math.sqrt, "abs": abs, "round": round,
        "log": math.log, "log10": math.log10,
        "sin": math.sin, "cos": math.cos, "tan": math.tan,
        "pi": math.pi, "e": math.e, "__builtins__": {}
    }
    try:
        result = eval(expression, safe_globals)
        return {"expression": expression, "result": result}
    except Exception as ex:
        return {"expression": expression, "error": str(ex)}


if __name__ == "__main__":
    print("🚀 Starting UtilityServer on http://localhost:8000/mcp/")
    mcp.run(transport="http", port=8000)

---

## 🌐 Section 2: Deploying with ngrok

The Responses API must reach your server over the internet. During development, **ngrok** creates a temporary public tunnel to your local server. In production you'd deploy to any cloud provider.

### Setup (run these in your terminal, not in the notebook)

**Terminal 1 — Start your FastMCP server:**
```bash
python mcp_responses_server.py
# Output: 🚀 Starting UtilityServer on http://localhost:8000/mcp/
```

**Terminal 2 — Expose it with ngrok:**
```bash
ngrok http 8000
# Output includes something like:
# Forwarding  https://a1b2-12-34-56-78.ngrok-free.app -> http://localhost:8000
```

**Copy the `https://` URL from ngrok** and set it below. The MCP endpoint path is `/mcp/` (FastMCP's default for Streamable HTTP).

In [2]:
# ── Set your server URL here ──────────────────────────────────────────────────
# Replace with your actual ngrok URL or deployed server URL.

NGROK_URL = "https://earring-rebuff-chaplain.ngrok-free.dev"   # ← paste your ngrok URL here
MCP_URL   = f"{NGROK_URL}/mcp/"                        # FastMCP HTTP endpoint

print(f"MCP server URL: {MCP_URL}")
print()
print("Before continuing, confirm your server is running:")
print(f"  curl {MCP_URL}  →  should return a 200 or 406 (method not allowed = server is up)")

MCP server URL: https://earring-rebuff-chaplain.ngrok-free.dev/mcp/

Before continuing, confirm your server is running:
  curl https://earring-rebuff-chaplain.ngrok-free.dev/mcp/  →  should return a 200 or 406 (method not allowed = server is up)


---

## 🚀 Section 3: Your First Responses API + MCP Call

### How the Responses API Processes an MCP Tool

```
1. You call client.responses.create() with tools=[{type: "mcp", server_url: ...}]
2. OpenAI contacts your server → calls tools/list
   → An mcp_list_tools item is added to the response output
3. Model sees available tools and decides to call one
   → An mcp_tool_call item appears in the output
4. OpenAI contacts your server → executes the tool call
   → mcp_tool_result is added
5. Model uses the result to form a natural language answer
   → A message item appears with output_text
```

You pay only for the tokens used — no extra fees per MCP call.

In [3]:
# ── The simplest possible Responses API + MCP call ────────────────────────────

response = openai_client.responses.create(
    model="gpt-4.1",
    tools=[
        {
            "type": "mcp",
            "server_label": "utility_server",   # A name you choose — used in output items
            "server_url": MCP_URL,
            "require_approval": "never",         # Skip approval for trusted servers
        }
    ],
    input="Roll 3 six-sided dice for me!",
)

print("Output text:", response.output_text)
print(f"\nTotal tokens used: {response.usage.total_tokens}")

Output text: You rolled three six-sided dice and got: 3, 3, and 2. The total is 8!

Total tokens used: 567


In [4]:
# ── Inspect the full output — see exactly what OpenAI did ─────────────────────
# The output contains structured items showing every step the API took.

print("Full output items:\n")
for item in response.output:
    print(f"  [{item.type}]")
    if item.type == "mcp_list_tools":
        tool_names = [t.name for t in item.tools]
        print(f"    Server: {item.server_label}")
        print(f"    Tools discovered: {tool_names}")
    elif item.type == "mcp_tool_call":
        print(f"    Tool called: {item.name}")
        print(f"    Arguments:   {item.arguments}")
    elif item.type == "message":
        print(f"    Role: {item.role}")
        for c in item.content:
            if hasattr(c, 'text'):
                print(f"    Text: {c.text[:100]}")
    print()

Full output items:

  [mcp_list_tools]
    Server: utility_server
    Tools discovered: ['roll_dice', 'get_weather', 'analyze_text', 'convert_units', 'calculate']

  [mcp_call]

  [message]
    Role: assistant
    Text: You rolled three six-sided dice and got: 3, 3, and 2. The total is 8!



In [17]:
# ── Test more tools ────────────────────────────────────────────────────────────

questions = [
    "What's the weather like in Tokyo and Manila right now?",
    "Convert 100 USD to PHP and also 75 km to miles.",
    "Analyze this text: 'The quick brown fox jumps over the lazy dog. The dog barked loudly at the fox.'",
    "What is the square root of 1764 plus 2 to the power of 8?",
]

for q in questions:
    print(f"{'='*65}")
    print(f"Q: {q}")
    resp = openai_client.responses.create(
        model="gpt-4.1",
        tools=[{"type": "mcp", "server_label": "utility",
                "server_url": MCP_URL, "require_approval": "never"}],
        input=q,
    )
    print(f"A: {resp.output_text}")
    # Show which tools were called
    calls = [i.name for i in resp.output if i.type == "mcp_call"]
    print(f"   (Tools called: {calls})")
    print()

Q: What's the weather like in Tokyo and Manila right now?
A: Here are the current weather conditions:

- **Tokyo:** 18°C, Overcast, 62% humidity, wind speed 20 kph
- **Manila:** 32°C, Partly Cloudy, 78% humidity, wind speed 15 kph

Let me know if you need more details or a forecast!
   (Tools called: ['get_weather', 'get_weather'])

Q: Convert 100 USD to PHP and also 75 km to miles.
A: - 100 USD is approximately 5,620 PHP.
- 75 kilometers is approximately 46.60 miles.
   (Tools called: ['convert_units', 'convert_units'])

Q: Analyze this text: 'The quick brown fox jumps over the lazy dog. The dog barked loudly at the fox.'
A: Here's an analysis of your text:

- Character count: 78
- Word count: 16
- Sentence count: 2
- Paragraph count: 1
- Estimated reading time: 5 seconds
- Top 5 most frequent words:
  1. fox (2)
  2. dog (2)
  3. quick (1)
  4. brown (1)
  5. jumps (1)
- Average words per sentence: 8.0

Let me know if you need a deeper analysis!
   (Tools called: ['analyze_text'])

Q

---

## ⚙️ Section 4: Responses API Features

### 4.1 — `require_approval`

By default the Responses API **pauses and asks for your approval** before each tool call — important for servers that might access sensitive data or have side effects. Once you trust a server, set `require_approval: "never"`.

In [18]:
# ── require_approval: "always" — inspect what approval requests look like ──────

response_with_approval = openai_client.responses.create(
    model="gpt-4.1",
    tools=[{
        "type": "mcp",
        "server_label": "utility",
        "server_url": MCP_URL,
        "require_approval": "always",   # ← pause before EVERY tool call
    }],
    input="Roll two dice.",
)

print("Output items when approval is required:\n")
approval_request_id = None
for item in response_with_approval.output:
    print(f"  [{item.type}]")
    if item.type == "mcp_approval_request":
        approval_request_id = item.id
        print(f"    ⏸️  Approval required for: {item.name}({item.arguments})")
        print(f"    Approval request ID: {item.id}")

print(f"\nResponse stopped at status: {response_with_approval.status}")

Output items when approval is required:

  [mcp_list_tools]
  [mcp_approval_request]
    ⏸️  Approval required for: roll_dice({"n_dice":2,"sides":6})
    Approval request ID: mcpr_0378b7a5001e6db1006a0e9ac1363481989a41a979191388fb

Response stopped at status: completed


In [19]:
# ── Approve the tool call and continue ───────────────────────────────────────
# When require_approval="always", you must send an mcp_approval_response
# to let the API continue. This lets you audit what data is sent to your server.

if approval_request_id:
    continued = openai_client.responses.create(
        model="gpt-4.1",
        tools=[{"type": "mcp", "server_label": "utility",
                "server_url": MCP_URL, "require_approval": "always"}],
        previous_response_id=response_with_approval.id,  # link to previous turn
        input=[{
            "type": "mcp_approval_response",
            "approve": True,                              # ✅ approve it
            "approval_request_id": approval_request_id,  # must match the request
        }],
    )
    print("After approval:")
    print(f"  {continued.output_text}")
else:
    print("No approval request found — try running the previous cell first.")

After approval:
  You rolled two dice: 4 and 4. The total is 8.


### 4.2 — `allowed_tools` — Filter Which Tools the Model Can See

In [20]:
# ── allowed_tools: expose only a subset of the server's tools ─────────────────
# The model only sees the tools you allow — even if the server exposes more.
# This is useful for restricting scope or reducing token usage.

# ← No filter — model sees ALL 5 tools
r_all = openai_client.responses.create(
    model="gpt-4.1",
    tools=[{"type": "mcp", "server_label": "utility",
            "server_url": MCP_URL, "require_approval": "never"}],
    input="What tools do you have available?",
)

# ← Filtered — model sees only 2 tools
r_filtered = openai_client.responses.create(
    model="gpt-4.1",
    tools=[{
        "type": "mcp",
        "server_label": "utility",
        "server_url": MCP_URL,
        "require_approval": "never",
        "allowed_tools": ["get_weather", "convert_units"],  # ← only these two
    }],
    input="What tools do you have available?",
)

# Count tools listed in mcp_list_tools output item
def count_tools(resp):
    for item in resp.output:
        if item.type == "mcp_list_tools":
            return [t.name for t in item.tools]
    return []

print("All tools visible:", count_tools(r_all))
print("Filtered tools:   ", count_tools(r_filtered))
print()
print("Model response (filtered):", r_filtered.output_text)

All tools visible: ['roll_dice', 'get_weather', 'analyze_text', 'convert_units', 'calculate']
Filtered tools:    ['get_weather', 'convert_units']

Model response (filtered): Here are the tools I currently have available:

### 1. **Image Input Capabilities**
- I can analyze and interpret images you upload—this includes reading text, recognizing objects, extracting information, and more.

### 2. **mcp_utility Plugin**
- **Weather Information:** I can provide current (simulated/demo) weather conditions for a specified city (temperature, humidity, wind speed, etc.).
- **Unit Conversion:** I can convert values between different units, including:
  - Length (e.g., kilometers to miles, meters to feet)
  - Weight (e.g., kilograms to pounds)
  - Temperature (e.g., Celsius to Fahrenheit)
  - Currency (e.g., USD to PHP, USD to EUR, etc.)

If you want to use any of these capabilities, just let me know!


### 4.3 — Tool List Caching with `previous_response_id`

The first call fetches the tool list from your server (adding an `mcp_list_tools` item). As long as this item exists in the conversation context, OpenAI **won't call your server's `tools/list` again** — reducing latency and your server's load.

Pass `previous_response_id` to keep the conversation context — including the cached tool list — across turns.

In [21]:
# ── Tool list caching — first call fetches, subsequent calls reuse ─────────────

import time

tool_config = [{
    "type": "mcp",
    "server_label": "utility",
    "server_url": MCP_URL,
    "require_approval": "never",
}]

# Turn 1 — tool list is fetched fresh
t0 = time.time()
r1 = openai_client.responses.create(
    model="gpt-4.1",
    tools=tool_config,
    input="What's the weather in Singapore?",
)
t1 = time.time()

# Turn 2 — uses previous_response_id → tool list is NOT re-fetched
t2 = time.time()
r2 = openai_client.responses.create(
    model="gpt-4.1",
    tools=tool_config,
    previous_response_id=r1.id,           # ← links conversation; reuses tool list
    input="And what about Sydney? Also convert 100 AUD... wait, do you have currency conversion?",
)
t3 = time.time()

def has_tool_fetch(resp):
    return any(i.type == "mcp_list_tools" for i in resp.output)

print(f"Turn 1 ({t1-t0:.1f}s): tool list fetched = {has_tool_fetch(r1)}")
print(f"  → {r1.output_text}")
print()
print(f"Turn 2 ({t3-t2:.1f}s): tool list fetched = {has_tool_fetch(r2)}")
print(f"  → {r2.output_text}")

Turn 1 (4.3s): tool list fetched = True
  → The current weather in Singapore is:
- Temperature: 30°C
- Condition: Thunderstorm
- Humidity: 90%
- Wind: 10 km/h

If you need a forecast or more details, let me know!

Turn 2 (4.7s): tool list fetched = False
  → The current weather in Sydney is:
- Temperature: 24°C
- Condition: Clear
- Humidity: 60%
- Wind: 22 km/h

Regarding currency conversion: Yes, I can convert currencies, but my supported conversions are currently between USD, PHP, EUR, and SGD. If you want to convert 100 AUD, I recommend converting first to USD and then to your desired currency, or you can specify which of the supported currencies you'd like to convert to. Would you like to try that?


---

## 🔄 Section 5: Multi-Turn Conversations

Chain multiple responses using `previous_response_id` to build a stateful conversation. The model remembers everything said in previous turns, including tool results.

In [22]:
# ── Multi-turn conversation with MCP tools ─────────────────────────────────────

tool_config = [{
    "type": "mcp",
    "server_label": "utility",
    "server_url": MCP_URL,
    "require_approval": "never",
}]

conversation = [
    "What's the weather in Tokyo and Manila? Give me a brief comparison.",
    "Thanks! Now convert the Tokyo temperature to Fahrenheit.",
    "Great. What would 50,000 Philippine Peso be in USD?",
    "Finally, roll me 2d20 (two 20-sided dice) for a game I'm playing.",
]

prev_id = None

for i, user_msg in enumerate(conversation, 1):
    print(f"\n{'='*65}")
    print(f"Turn {i} — User: {user_msg}")
    print(f"{'='*65}")

    kwargs = {
        "model": "gpt-4.1",
        "tools": tool_config,
        "input": user_msg,
    }
    if prev_id:
        kwargs["previous_response_id"] = prev_id  # link to previous turn

    response = openai_client.responses.create(**kwargs)

    # Show which tools were called
    calls = [(i.name, i.arguments) for i in response.output if i.type == "mcp_tool_call"]
    if calls:
        for name, args in calls:
            print(f"  🔧 Tool called: {name}({args})")

    print(f"\n  Assistant: {response.output_text}")

    prev_id = response.id  # carry forward for next turn


Turn 1 — User: What's the weather in Tokyo and Manila? Give me a brief comparison.

  Assistant: Here's a brief weather comparison between Tokyo and Manila:

- **Tokyo:** 18°C, Overcast, 62% humidity, Wind 20 kph
- **Manila:** 32°C, Partly Cloudy, 78% humidity, Wind 15 kph

**Summary:** Tokyo is much cooler and more overcast, while Manila is significantly warmer, more humid, and has partly cloudy skies. Winds are slightly stronger in Tokyo.

Turn 2 — User: Thanks! Now convert the Tokyo temperature to Fahrenheit.

  Assistant: The temperature in Tokyo, 18°C, is equivalent to 64.4°F.

Turn 3 — User: Great. What would 50,000 Philippine Peso be in USD?

  Assistant: ₱50,000 Philippine Peso is approximately $889.68 USD.

Turn 4 — User: Finally, roll me 2d20 (two 20-sided dice) for a game I'm playing.

  Assistant: You rolled two 20-sided dice and got: 19 and 11. The total is 30. Good luck with your game!


---

## 🏭 Section 6: Production Patterns

### 6.1 — Multiple MCP Servers in One Request

The Responses API accepts multiple entries in the `tools` array. The model can call tools from any of them in a single response.

In [28]:
# ── Multiple MCP servers ───────────────────────────────────────────────────────
# In production you might have separate specialized servers.
# Here we point to the same server twice with different labels and allowed_tools
# to simulate two separate server roles.

response = openai_client.responses.create(
    model="gpt-4.1",
    tools=[
        {
            "type": "mcp",
            "server_label": "weather_server",       # ← logical name 1
            "server_url": MCP_URL,
            "require_approval": "never",
            "allowed_tools": ["get_weather"],        # only weather
        },
        {
            "type": "mcp",
            "server_label": "math_server",           # ← logical name 2
            "server_url": MCP_URL,
            "require_approval": "never",
            "allowed_tools": ["calculate", "convert_units"],  # only math
        },
    ],
    input="What's the weather in London? Also, how many meters is the Eiffel Tower "
          "(330 meters) in feet, and what's 330 squared?",
)

print(response.output_text)
print()

# Show which server/tool was called for each invocation
print("Tool calls:")
for item in response.output:
    if item.type == "mcp_call":
        print(f"  [{item.server_label}] {item.name}({item.arguments})")

Here's the information you requested:

- The weather in London is currently 12°C, rainy, with 85% humidity and winds at 25 kph.
- The Eiffel Tower's height of 330 meters is approximately 1082.68 feet.
- 330 squared is 108,900 (there was a calculation mishap earlier; the correct value is 330 × 330 = 108,900).

Let me know if you need more conversions or details!

Tool calls:
  [weather_server] get_weather({"city":"London","units":"celsius"})
  [math_server] convert_units({"value":330,"from_unit":"m","to_unit":"ft"})
  [math_server] calculate({"expression":"330^2"})


### 6.2 — Mixing MCP with Regular Function Calling

You can combine `type: "mcp"` tools with `type: "function"` tools in the same request. The model decides which to use.

In [29]:
# ── Mix: MCP server + a local function tool ───────────────────────────────────
# Use case: MCP server handles external calls; local function handles
# internal logic that can't be exposed publicly.

import json

# A local function that shouldn't be on the public server
INTERNAL_DB = {
    "PROD-001": {"name": "Widget A", "stock": 142, "warehouse": "Manila"},
    "PROD-002": {"name": "Gadget B", "stock": 37,  "warehouse": "Cebu"},
}

def check_inventory(product_id: str) -> dict:
    return INTERNAL_DB.get(product_id, {"error": f"{product_id} not found"})


response = openai_client.responses.create(
    model="gpt-4.1",
    tools=[
        # MCP tool (remote server)
        {
            "type": "mcp",
            "server_label": "utility",
            "server_url": MCP_URL,
            "require_approval": "never",
            "allowed_tools": ["convert_units"],
        },
        # Local function tool (stays in your code)
        {
            "type": "function",
            "name": "check_inventory",
            "description": "Check stock levels for a product by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_id": {"type": "string",
                                   "description": "Product ID (e.g. PROD-001)"}
                },
                "required": ["product_id"]
            }
        }
    ],
    input="How much stock do we have of PROD-001? Also, the warehouse is in Manila — "
          "how far is that from Cebu in miles? (Manila to Cebu is about 570 km)",
)

# Handle the agentic loop — process any function calls
messages = list(response.output)
function_outputs = []

for item in messages:
    if item.type == "function_call":
        print(f"🔧 Local function call: {item.name}({item.arguments})")
        args = json.loads(item.arguments)
        result = check_inventory(**args)
        function_outputs.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(result)
        })

if function_outputs:
    # Send function results back and get final answer
    final = openai_client.responses.create(
        model="gpt-4.1",
        tools=[
            {"type": "mcp", "server_label": "utility", "server_url": MCP_URL,
             "require_approval": "never", "allowed_tools": ["convert_units"]},
            {"type": "function", "name": "check_inventory",
             "description": "Check stock levels.",
             "parameters": {"type": "object", "properties": {"product_id": {"type": "string"}}, "required": ["product_id"]}}
        ],
        previous_response_id=response.id,
        input=function_outputs,
    )
    print(f"\nFinal answer: {final.output_text}")
else:
    # No local function calls — model answered directly
    print(f"\nAnswer: {response.output_text}")

🔧 Local function call: check_inventory({"product_id":"PROD-001"})

Final answer: You currently have 142 units of "Widget A" (PROD-001) in stock at the Manila warehouse.

The distance from Manila to Cebu is approximately 354 miles (about 570 km). If you need more specific logistics details or inventory for another product, let me know!


### 6.3 — Using Official MCP Servers (GitHub, Stripe, etc.)

Major services now host official MCP servers you can point the Responses API directly at — no setup required.

In [ ]:
# ── Connecting to the official GitHub MCP server ──────────────────────────────
# GitHub hosts an official MCP server at https://api.githubcopilot.com/mcp/
# You need a GitHub personal access token (classic) with 'repo' scope.

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "ghp_your_token_here")

# Uncomment to use (requires a valid GitHub token):
"""
response = openai_client.responses.create(
    model="gpt-4.1",
    tools=[{
        "type": "mcp",
        "server_label": "github",
        "server_url": "https://api.githubcopilot.com/mcp/",
        "require_approval": "never",
        "headers": {
            "Authorization": f"Bearer {GITHUB_TOKEN}"
        },
        "allowed_tools": ["get_file_contents", "list_commits", "search_repositories"],
    }],
    input="What are the 5 most recent commits to the openai/openai-python repository?",
)
print(response.output_text)
"""

# ── Other official MCP servers ─────────────────────────────────────────────────
print("""
Official MCP Servers you can use with the Responses API:

  GitHub        https://api.githubcopilot.com/mcp/
                Auth: Bearer <github_personal_access_token>

  Stripe        https://mcp.stripe.com
                Auth: Bearer <stripe_api_key>

  Shopify       https://mcp.shopify.com/v1
                Auth: Bearer <shopify_access_token>

  Cloudflare    https://mcp.cloudflare.com
                Auth: Bearer <cloudflare_api_token>

  Jira          https://mcp.atlassian.com/mcp
                Auth: Bearer <atlassian_api_token>

Use these by pointing server_url at the service's MCP endpoint and passing
your API key in the Authorization header.
""")

---

## 🎉 Summary

### The Full Picture

```
┌─────────────────────────────────────────────────────────────────────────┐
│                      Responses API + MCP Flow                          │
│                                                                         │
│  client.responses.create(                                               │
│      model="gpt-4.1",                                                  │
│      tools=[{                                                           │
│          "type": "mcp",                                                 │
│          "server_label": "my_server",     ← your name for this server  │
│          "server_url": "https://...",     ← must be public HTTPS        │
│          "require_approval": "never",     ← or "always" for prod review │
│          "allowed_tools": ["tool1"],      ← optional filter             │
│          "headers": {"Authorization": …} ← optional auth               │
│      }],                                                                │
│      input="User message",                                              │
│      previous_response_id=prev_id,        ← optional: chain turns      │
│  )                                                                      │
│                                                                         │
│  Response output items:                                                 │
│    mcp_list_tools   — tool schemas (cached after first call)            │
│    mcp_tool_call    — what the model decided to call                    │
│    mcp_tool_result  — what your server returned                         │
│    message          — model's final natural language answer             │
└─────────────────────────────────────────────────────────────────────────┘
```

### Responses API vs. Chat Completions for MCP

| Feature | Chat Completions | Responses API |
|---|---|---|
| MCP support | ❌ | ✅ Native |
| Function calling | ✅ | ✅ |
| Tool loop | Your code | OpenAI handles |
| Conversation state | You manage messages | `previous_response_id` |
| Streaming | ✅ | ✅ |
| Multiple MCP servers | — | ✅ |
| Approval gating | — | ✅ |
| Tool list caching | — | ✅ Automatic |

### What Your FastMCP Server Needs

```python
mcp.run(transport="http", port=8000)  # Must be HTTP transport
# → Endpoint: http://localhost:8000/mcp/
# → Must be publicly reachable (ngrok for dev, cloud for prod)
# → Only TOOLS are used — resources and prompts are ignored
```

### 📚 References
- 📖 [OpenAI MCP & Connectors Guide](https://developers.openai.com/api/docs/guides/tools-connectors-mcp)
- 📖 [FastMCP × OpenAI Integration](https://gofastmcp.com/integrations/openai)
- 📖 [OpenAI Responses API Reference](https://developers.openai.com/api/reference/responses)
- 📖 [OpenAI MCP Cookbook](https://cookbook.openai.com/examples/mcp/mcp_tool_guide)